In [2]:
import requests
import time
from mal_scraper import get_user_anime_list
import logging
import pandas as pd
import numpy as np

In [3]:
logging.getLogger('jikanpy').setLevel(logging.CRITICAL)
logging.getLogger().setLevel(logging.CRITICAL)

In [4]:
delay = 1
maxUserSize = 50000

In [5]:
def getRandomUser():
    time.sleep(delay)
    url = "https://api.jikan.moe/v4/random/users"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            return data['data']
        elif response.status_code == 429:
            print("Rate limited! Waiting before retry...")
            time.sleep(0.1)
            return getRandomUser()
        else:
            print(f"Error: {response.status_code}")
            return None
            
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

In [6]:
username = "Exarfate"
print(username)

Exarfate


In [7]:
def getReviewsFromUser(username):
    anime_list = get_user_anime_list(username)
    if anime_list is None: return []
    anime_list = [anime for anime in anime_list if anime['score'] > 0]
    return anime_list

In [8]:
anime_list = getReviewsFromUser(username)
print(anime_list)
print(len(anime_list))

[{'name': 'Code Geass: Hangyaku no Lelouch', 'id_ref': 1575, 'consumption_status': <ConsumptionStatus.completed: 'COMPLETED'>, 'is_rewatch': False, 'score': 10, 'progress': 25, 'start_date': None, 'finished_date': None}, {'name': 'Code Geass: Hangyaku no Lelouch R2', 'id_ref': 2904, 'consumption_status': <ConsumptionStatus.completed: 'COMPLETED'>, 'is_rewatch': False, 'score': 10, 'progress': 25, 'start_date': None, 'finished_date': None}]
2


In [9]:
url = "https://api.jikan.moe/v4/top/anime"
while True:
    response = requests.get(url)
    
    time.sleep(delay)
    
    data = response.json()
    if not 'status' in data or data['status'] != 500:
        break


In [18]:
url = "https://api.jikan.moe/v4/top/anime"
popular_anime_ids = []
for page in range(1, 20):
    params = {
        "filter": "bypopularity",
        "page": page,
    }
    response = requests.get(url, params=params)
    time.sleep(delay)
    data = response.json()
    # print(data)
    animes = data['data']
    popular_anime_ids += [anime['mal_id'] for anime in animes]

print(popular_anime_ids)
print(len(popular_anime_ids))

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [10]:
def get_anime_reviews_jikan(anime_id, page):
    """
    Get reviews for an anime using Jikan API
    """
    url = f"https://api.jikan.moe/v4/anime/{anime_id}/reviews"
    params = {
        "page": page,
    }
    
    response = requests.get(url, params=params)
    time.sleep(delay)
    
    if response.status_code == 200:
        return response.json()
    else:
        return None

In [1]:
number = 1
t0 = time.time()
distinctUsers = set()
to_graph = [0]
for animeId in popular_anime_ids:
    page = 1
    while len(distinctUsers) < maxUserSize:
    # for page in range(1, 10):        
        data = get_anime_reviews_jikan(animeId, page)
        # print(data)
        for elem in data['data']:
            # print(elem)
            for i, review in enumerate(data['data'], 1):
                # print(f"\nReview {i}:")
                # print(f"User: {review['user']['username']}")
                # print(f"Score: {review['score']}")
                distinctUsers.add(review['user']['username'])

        if not data['pagination']['has_next_page']:
            print(f"{number} ended at: {page} with total of {len(distinctUsers)} users after {time.time() - t0} seconds")
            break
        page += 1
    to_graph.append(len(distinctUsers))
    number += 1

NameError: name 'time' is not defined

In [ ]:
print(len(distinctUsers))

In [ ]:
usernames = list(distinctUsers)

In [ ]:
rows = []
rowsAtLeast5 = []
usernamesAtLeast5 = []
for username in usernames:
    animeList = getReviewsFromUser(username)
    shortedAnimeList = [{
            'id_ref': entry['id_ref'],
            'name': entry['name'],
            'score': entry['score'],
        } 
        for entry in animeList]
    # print(animeList)
    # print(shortedAnimeList)
    # print(len(animeList))
    rows.append(shortedAnimeList)
    if len(shortedAnimeList) > 4:
        rowsAtLeast5.append(shortedAnimeList)
        usernamesAtLeast5.append(username)

In [ ]:
with open("usernames.txt", "w") as f:
    for item in distinctUsers:
        f.write(f"{item}\n")

In [ ]:
df = pd.DataFrame([
    {f"{item['id_ref']}_{item['name']}": item['score'] for item in userAnime}
    for userAnime in rows
], index=usernames)

df = df.fillna(0)

df

,31646_3-gatsu no Lion,35180_3-gatsu no Lion 2nd Season,9776_A-Channel,12291_Acchi Kocchi,36904_Aggressive Retsuko (ONA),37985_Aggressive Retsuko (ONA) 2nd Season,40215_Aggressive Retsuko (ONA) 3rd Season,45489_Aggressive Retsuko (ONA) 4th Season,50598_Aggressive Retsuko (ONA) 5th Season,16201_Aku no Hana,...,49165_Bright: Samurai Soul,6671_Byeolnala Samchongsa,49163_Exception,9750_Itsuka Tenma no Kuro Usagi,598_Jinzou Ningen Kikaider The Animation,41168_Nakitai Watashi wa Neko wo Kaburu,816_Nanako Kaitai Shinsho,9303_Noel no Fushigi na Bouken,1226_Seihou Tenshi Angel Links,51347_Tekken: Bloodline
Micsupreeme,8.0,9.0,7.0,5.0,7.0,7.0,7.0,7.0,7.0,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TheAnimeGeneral,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
lawlmartz,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
DollFishu,8.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
kasser,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
KingFlabadingdon,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
genesic123,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
hjlee,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Nikolekoleta,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
IamWEB,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
df2 = pd.DataFrame([
    {f"{item['id_ref']}_{item['name']}": item['score'] for item in userAnime}
    for userAnime in rowsAtLeast5
], index=usernamesAtLeast5)

df2 = df2.fillna(0)

df2

,31646_3-gatsu no Lion,35180_3-gatsu no Lion 2nd Season,9776_A-Channel,12291_Acchi Kocchi,36904_Aggressive Retsuko (ONA),37985_Aggressive Retsuko (ONA) 2nd Season,40215_Aggressive Retsuko (ONA) 3rd Season,45489_Aggressive Retsuko (ONA) 4th Season,50598_Aggressive Retsuko (ONA) 5th Season,16201_Aku no Hana,...,49165_Bright: Samurai Soul,6671_Byeolnala Samchongsa,49163_Exception,9750_Itsuka Tenma no Kuro Usagi,598_Jinzou Ningen Kikaider The Animation,41168_Nakitai Watashi wa Neko wo Kaburu,816_Nanako Kaitai Shinsho,9303_Noel no Fushigi na Bouken,1226_Seihou Tenshi Angel Links,51347_Tekken: Bloodline
Micsupreeme,8.0,9.0,7.0,5.0,7.0,7.0,7.0,7.0,7.0,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TheAnimeGeneral,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
lawlmartz,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
DollFishu,8.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
kasser,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
KingFlabadingdon,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
genesic123,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Nikolekoleta,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
IamWEB,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Absolute_Spider,0.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
df.to_csv('anime_df.csv')  # index=False removes row numbers
df

,31646_3-gatsu no Lion,35180_3-gatsu no Lion 2nd Season,9776_A-Channel,12291_Acchi Kocchi,36904_Aggressive Retsuko (ONA),37985_Aggressive Retsuko (ONA) 2nd Season,40215_Aggressive Retsuko (ONA) 3rd Season,45489_Aggressive Retsuko (ONA) 4th Season,50598_Aggressive Retsuko (ONA) 5th Season,16201_Aku no Hana,...,49165_Bright: Samurai Soul,6671_Byeolnala Samchongsa,49163_Exception,9750_Itsuka Tenma no Kuro Usagi,598_Jinzou Ningen Kikaider The Animation,41168_Nakitai Watashi wa Neko wo Kaburu,816_Nanako Kaitai Shinsho,9303_Noel no Fushigi na Bouken,1226_Seihou Tenshi Angel Links,51347_Tekken: Bloodline
Micsupreeme,8.0,9.0,7.0,5.0,7.0,7.0,7.0,7.0,7.0,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TheAnimeGeneral,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
lawlmartz,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
DollFishu,8.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
kasser,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
KingFlabadingdon,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
genesic123,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
hjlee,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Nikolekoleta,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
IamWEB,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
print(len(usernames))

60
